# Reward models — ARCHIVED

**Status: superseded by the selector and neural work. Do not build on this.**

Per-severity reward models: sample actions uniformly, record state and reward, fit a network
to predict reward, then take the argmax over candidate actions. It replaced TD3 by covering
the whole action space instead of whatever an untrained actor proposed.

**The result worth remembering:** cross-evaluation showed each model predicts best on the
severity it was trained for — but the models almost all *select the same action* regardless.
Specialisation was real in reward prediction and never reached the control decision.

The datasets it produced are still live: `datasets/pd*.json`, about 35 minutes of MATLAB each.

# Reward-model approach

Instead of training a policy with TD3, learn a **reward model**: a network mapping
`(state, action)` to predicted reward. The controller is then `argmax` over candidate
actions.

Why this instead of TD3:

- **No exploration failure.** Actions are sampled uniformly by us, so the training data
  covers the whole space. TD3's untrained actor only ever produced 1600-3500 uA/cm2, which
  is why it never found the good low-amplitude region.
- **MATLAB runs once per severity, not once per architecture.** The same dataset trains
  32, 64, or any other size.
- **Fitting takes seconds.** Comparing architectures becomes a supervised-learning
  question, not a 14-hour experiment.

| # | Step | MATLAB | Time |
|---|---|---|---|
| 1 | Setup | no | instant |
| 2 | Start MATLAB | yes | ~1 min |
| 3 | Collect data (3 severities) | yes | ~45-70 min |
| 4 | Fit all architectures | no | ~1 min |
| 5 | Cross-evaluation matrix | no | instant |
| 6 | Which action does each pick | no | instant |
| 7 | Validate best action in MATLAB | yes | ~10 min |

## 1. Setup

In [ ]:
import importlib
import json
import numpy as np
from pathlib import Path
from dataclasses import replace

import dbs_core, dbs_checks, dbs_plots, dbs_reward_model as rm
for m in (dbs_core, dbs_checks, dbs_plots, rm):
    importlib.reload(m)

from dbs_core import Config, MatlabEnv, MockEnv

assert dbs_checks.versions()
print("reward model module:", rm.__version__)

## 2. Start MATLAB

In [ ]:
cfg = Config(
    pd_value=0.4,
    tmax=1500,
    steps_per_episode=15,
    rms_max=1200.0,
    run_dir="runs",
    tag="rm",
)
menv = MatlabEnv(cfg, mat_path="bgn_vars.mat")
print("engine started")

## 3. Collect

Uniform random actions, each held for `hold` steps so the biomarker settles before the
sample is recorded (the probe data showed ~3 steps of turn-on transient).

`n_samples=400, hold=4` is about 1600 MATLAB steps per severity — roughly 23 min each at
0.88 s/step. Raw `sgis` and `rms` are stored, so rewards can be recomputed under different
weights without re-collecting.

In [ ]:
PD_VALUES = [0.3, 0.6, 1.0]
N_SAMPLES = 400
HOLD = 4

datasets = {}
for pd in PD_VALUES:
    c = replace(cfg, pd_value=pd)
    menv.cfg = c
    ok, samples, _ = dbs_checks.live(menv, c, verbose=False)
    print(f"pd={pd}: env checks {'PASS' if ok else 'FAIL — treat with caution'}")
    datasets[f"pd{pd}"] = rm.collect(menv, c, n_samples=N_SAMPLES, hold=HOLD)

In [ ]:
PD_VALUES = [0.3, 0.6, 1.0]
N_SAMPLES = 400
HOLD = 4

c = replace(cfg, pd_value=1.0)
menv.cfg = c
ok, samples, _ = dbs_checks.live(menv, c, verbose=False)
print(f"pd=1.0: env checks {'PASS' if ok else 'FAIL'}")
rm.collect(menv, c, n_samples=N_SAMPLES, hold=HOLD)

## 3b. Reload saved data (skip collection if already done)

In [ ]:
datasets = {}
for p in sorted(Path("datasets").glob("pd*_n*.json")):
    key = p.stem.split("_")[0]
    datasets[key] = json.loads(p.read_text())
    print(f"{key}: {len(datasets[key])} samples")

## 4. Fit every architecture

Seconds per model. Val R2 is the number that matters — how much of the reward variance
the model explains on held-out data.

In [ ]:
HIDDEN_DIMS = [16, 32, 64, 128]
SEEDS = [0, 1, 2]

models, fit_stats = {}, []
for name, rows in sorted(datasets.items()):
    print(f"\n{name}  (n={len(rows)})")
    for h in HIDDEN_DIMS:
        r2s = []
        for seed in SEEDS:
            m, st = rm.fit(rows, cfg, hidden_dim=h, seed=seed, verbose=False)
            r2s.append(st["val_r2"])
            if seed == 0:
                models[f"{name}_h{h}"] = m
                fit_stats.append({**st, "dataset": name})
        print(f"  hidden={h:<4} params={st['n_params']:<6} "
              f"val R2 {np.mean(r2s):.3f} +/- {np.std(r2s):.3f}")

## 5. Cross-evaluation matrix

**This is the deliverable.** Each model scored against every severity's data.

- Strong diagonal, weak off-diagonal -> specialization is real, a selector is worth building
- Flat everywhere -> one model generalizes, no selector needed

In [ ]:
for h in HIDDEN_DIMS:
    sub = {k.replace(f"_h{h}", ""): v for k, v in models.items() if k.endswith(f"_h{h}")}
    if len(sub) < 2:
        continue
    print(f"\n=== hidden={h} ===")
    rm.cross_evaluate(sub, datasets, cfg)

## 6. Which action does each model choose

If the models pick genuinely different actions by severity, that is the mechanism a
selector would exploit. If they all pick the same one, specialization is not actionable
even if the R2 matrix looks specialized.

In [ ]:
states = [np.array([r[f"s{i}"] for i in range(cfg.state_dim)], dtype=np.float32)
          for r in datasets[f"pd{PD_VALUES[1]}"][:60]]

for h in HIDDEN_DIMS:
    sub = {k.replace(f"_h{h}", ""): v for k, v in models.items() if k.endswith(f"_h{h}")}
    if len(sub) < 2:
        continue
    print(f"\n=== hidden={h} ===")
    rm.action_agreement(sub, cfg, states)

In [ ]:
import numpy as np

for name, rows in sorted(datasets.items()):
    print(f"\n=== {name} ===")
    print(f"{'amp bin':<14} {'n':>4} {'mean sgis':>10} {'mean reward':>12}")
    bins = [(0, 500), (500, 1500), (1500, 2500), (2500, 3500), (3500, 5000)]
    for lo, hi in bins:
        sel = [r for r in rows if lo <= r["amp"] < hi]
        if not sel:
            continue
        r1 = [np.clip((r["sgis"] - cfg.sgis_min) / (cfg.sgis_max - cfg.sgis_min), 0, 1) for r in sel]
        r2 = [np.clip(r["rms"] / cfg.rms_max, 0, 1) for r in sel]
        rew = [-cfg.w_biomarker * a - cfg.w_energy * b for a, b in zip(r1, r2)]
        print(f"{lo:>5}-{hi:<8} {len(sel):>4} {np.mean([r['sgis'] for r in sel]):>10.0f} "
              f"{np.mean(rew):>12.4f}")

In [ ]:
import numpy as np

fbins = [(0, 60), (60, 110), (110, 145), (145, 168), (168, 186)]
abins = [(0, 500), (500, 1500), (1500, 3000), (3000, 5000)]

for name, rows in sorted(datasets.items()):
    print(f"\n=== {name} — mean sgis ===")
    print(f"{'freq':<12}" + "".join(f"{f'{lo}-{hi}':>12}" for lo, hi in abins))
    for flo, fhi in fbins:
        line = f"{f'{flo}-{fhi}Hz':<12}"
        for alo, ahi in abins:
            sel = [r for r in rows if flo <= r["freq"] < fhi and alo <= r["amp"] < ahi]
            line += f"{np.mean([r['sgis'] for r in sel]):>12.0f}" if len(sel) >= 3 else f"{'—':>12}"
        print(line)

## 7. Validate in MATLAB

The model's chosen action, run for real, against the fixed baselines. This is what says
whether the reward model beats both TD3 and open-loop DBS.

In [ ]:
for pd in PD_VALUES:
    c = replace(cfg, pd_value=pd)
    menv.cfg = c
    model = models[f"pd{pd}_h64"]
    s = menv.reset()
    (freq, amp), pred, _ = rm.best_action(model, s, c)
    print(f"\npd={pd}: model picks {freq}Hz/{amp}uA (predicted reward {pred:+.4f})")
    fixed = dbs_checks.baseline(menv, c, actions=[(0, 0), (freq, amp), (180, 1000)],
                                n_episodes=3)

## Shut down

In [ ]:
menv.close()
print("engine closed")